In [ ]:
from probe_design import *
from probe_design_utils import *
from probe_design_qc import *

## Example 1: Multidomain dCas9 Epigenome Editor Custom Probes

In [ ]:
fasta_reference = "references/hMOI_Effectors.fasta"

xlsx_sheet_to_fasta(
    xlsx_path="references/Extended_data_table_2.xlsx",
    sheet_name="2B_selected_effectors",
    id_col="header",
    seq_col="sequence",
    output_fasta=fasta_reference
)

nominated_sites  = "probe_nomination/hMOI_Effectors_probes.fasta"
nominated_probes  = "probe_nomination/hMOI_Effectors_lhs_rhs_probes.fasta"

nominate_probes(fasta_reference)
lhs_rhs_probe_binding_site_split(nominated_sites, nominated_probes)

In [ ]:
blast_results = "probe_nomination/hMOI_Effectors_blast.txt"

subprocess.run(
    ["scripts/custom_probe_blast.sh", nominated_probes, blast_results, "-t", "8"],
    check=True,      
    text=True,
)

In [ ]:
probe_annotations_csv = 'probe_nomination/hMOI_Effectors_annotations.csv'
blast_probe_annotation = 'probe_nomination/hMOI_Effectors_blast_annotations.csv'

lhs_df, rhs_df = read_blast_results(blast_results)
process_lhs_rhs(lhs_df,rhs_df,probe_annotations_csv, blast_probe_annotation)

In [ ]:
nominated_probes = 'probe_nomination/hMOI_Effectors_top_probes.csv'
nominate_top_probe_set(blast_probe_annotation, nominated_probes)

In [ ]:
R2N = "CGGAGATGTGTATAAGAGACAG"

construct_custom_probe_opool(
    nominated_probe_csv="probe_nomination/hMOI_Effectors_top_probes.csv",
    pool_name="hMOI_Effectors",
    lhs_r2_adaptor=R2N,
    n_barcodes=16,
)

## Example 2: Masking Regions, Missense Variant T-cell Activating ORFs

In [ ]:
xlsx_sheet_to_fasta(
    xlsx_path="references/Extended_data_table_3.xlsx",
    sheet_name="4E_Missense_ORF_Sequences",
    id_col="id",
    seq_col="sequence",
    output_fasta="references/Missense_ORFs.fasta"
)

nominate_probes("references/Missense_ORFs.fasta")
nominated_sites  = "probe_nomination/Missense_ORFs_probes.fasta"
nominated_probes = "probe_nomination/Missense_ORFs_lhs_rhs_probes.fasta"

lhs_rhs_probe_binding_site_split(nominated_sites,nominated_probes)

blast_results = "probe_nomination/Missense_ORFs_blast.txt"

subprocess.run(
    ["scripts/custom_probe_blast.sh", nominated_probes, blast_results],
    check=True,      
    text=True,
)

probe_annotations_csv = 'probe_nomination/Missense_ORFs_annotations.csv'
blast_probe_annotation_output = 'probe_nomination/Missense_ORFs_blast_annotations.csv'

lhs_df, rhs_df = read_blast_results(blast_results)
process_lhs_rhs(lhs_df,rhs_df,probe_annotations_csv,blast_probe_annotation_output)

In [ ]:
gapfill_file = 'references/Extended_data_table_3.xlsx'  
filtered_blast_annotation = 'probe_nomination/Missense_ORF_Sequences_blast_annotations_filtered.csv'  
gapfill_sheet = '4G_Gapfill_Position_Lengths'

filter_probes_and_save(blast_probe_annotation_output, gapfill_file, filtered_blast_annotation, gapfill_sheet)

nominated_probes = 'probe_nomination/Missense_ORF_top_probes.csv'
nominate_top_probe_set(filtered_blast_annotation, nominated_probes)


R2N = "CGGAGATGTGTATAAGAGACAG"

construct_custom_probe_opool(
    nominated_probe_csv="probe_nomination/Missense_ORF_top_probes.csv",
    pool_name="Missense_ORF",
    lhs_r2_adaptor=R2N,
    n_barcodes=4,
)

## Probe plotting and inspection

In [ ]:
fd_seq = pd.DataFrame([(rec.id, str(rec.seq).upper()) for rec in SeqIO.parse("references/hMOI_Effectors.fasta", "fasta")], columns=["header", "sequence"],)
fd_blast_scored = calculate_priority_score(pd.read_csv('probe_nomination/hMOI_Effectors_blast_annotations.csv'))
fd_top = pd.read_csv('probe_nomination/hMOI_Effectors_top_probes.csv')

target_id = "C32" 
fd_sequence     = fd_seq.loc[fd_seq["header"] == target_id, "sequence"].values[0]
fd_blast_target = fd_blast_scored[fd_blast_scored["target"] == target_id].copy()
fd_top_target   = fd_top[fd_top["target"] == target_id].copy()
Path("plots").mkdir(exist_ok=True)
fig = plot_metrics(fd_sequence, seq_id="C32", blast_df=fd_blast_target, top_df=fd_top_target, svg_path=f"plots/{target_id}_QC.svg")